# TabFM Beginner Walkthrough

An interactive companion to [`docs/03-first-run.md`](../docs/03-first-run.md) and
[`docs/04-core-concepts.md`](../docs/04-core-concepts.md). By the end of this notebook you will have:

1. Made your first TabFM prediction (zero-shot, no training loop).
2. Seen *why* `.fit()` doesn't train a model in the usual sense.
3. Compared TabFM's **default** preset against its **ensemble** preset on the same data.

**Prerequisites:** finish [`docs/00-overview.md`](../docs/00-overview.md) through
[`docs/02-installation.md`](../docs/02-installation.md) first — this notebook assumes TabFM is installed
and (optionally) that you've read what "in-context learning" and "zero-shot" mean.

> **License reminder:** TabFM's released weights are under the **TabFM Non-Commercial License v1.0** —
> research/evaluation only. See [`docs/09-faq.md#licensing`](../docs/09-faq.md#licensing) before any commercial use.

## Step 0 — Setup

We auto-detect a GPU, but only use it if it has at least 12 GiB of VRAM. TabFM's weights alone occupy
several GB, and smaller GPUs run out of memory during prediction — see
[`docs/08-troubleshooting.md#cuda-out-of-memory`](../docs/08-troubleshooting.md#cuda-out-of-memory)
for exactly why. CPU is always a safe fallback, just slower.

In [ ]:
from __future__ import annotations

import os
import time

import torch
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

from tabfm import TabFMClassifier
from tabfm import tabfm_v1_0_0_pytorch as tabfm_v1_0_0

DEVICE = os.environ.get("TABFM_DEVICE", "auto")
if DEVICE == "auto":
    DEVICE = "cpu"
    if torch.cuda.is_available():
        gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        DEVICE = "cuda" if gpu_mem_gb >= 12.0 else "cpu"

# Optional: point this at a local checkpoint directory to skip the download.
# See docs/08-troubleshooting.md#pytorch-model-bin-not-found if load() fails below.
CHECKPOINT_PATH = os.environ.get("TABFM_CHECKPOINT_PATH") or None

print(f"Using device: {DEVICE}")

## Step 1 — Load a real, small dataset

[`load_breast_cancer`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html)
ships with scikit-learn: 569 rows, 30 numeric features (measurements from tumor scans),
binary target (malignant vs. benign). No download needed for this part.

We hold out 20% as a test set — see
[`docs/01-prerequisites.md`](../docs/01-prerequisites.md) if "why hold out a test set" isn't obvious yet.

In [ ]:
X, y = load_breast_cancer(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train rows: {len(X_train)} | Test rows: {len(X_test)} | Features: {X.shape[1]}")

**Expected output:**
```
Train rows: 455 | Test rows: 114 | Features: 30
```

## Step 2 — Load TabFM's pre-trained weights

This is a **frozen** model — nothing about it changes based on your data. The first time you run this
cell without `CHECKPOINT_PATH` set, it downloads TabFM's classification weights from Hugging Face
(several GB, one-time — see [`docs/02-installation.md`](../docs/02-installation.md)).

> If you hit `FileNotFoundError: ... pytorch_model.bin`, that is a known upstream packaging issue,
> not something you did wrong — see
> [`docs/08-troubleshooting.md#pytorch-model-bin-not-found`](../docs/08-troubleshooting.md#pytorch-model-bin-not-found)
> for the one-command fix (`scripts/fetch_tabfm_weights.py`).

In [ ]:
model = tabfm_v1_0_0.load(
    model_type="classification",
    device=DEVICE,
    checkpoint_path=CHECKPOINT_PATH,
)
print("Model loaded.")

## Step 3 — Fit and predict: the default preset

Watch closely: `.fit()` here does **not** train a neural network on your data. It only prepares
label/feature encoders and stores `X_train`/`y_train` as *context* — the frozen model reads that
context at prediction time and reasons over it via in-context learning. See
[`docs/04-core-concepts.md §1`](../docs/04-core-concepts.md#1-in-context-learning-precisely) for the full
explanation of why this isn't "training" in the usual sense.

In [ ]:
clf = TabFMClassifier(model=model, random_state=42)

start = time.perf_counter()
clf.fit(X_train, y_train.to_numpy())

# TabFM returns predictions as an object-dtype array — cast before scoring.
# See docs/08-troubleshooting.md#object-dtype-predictions for why this is needed.
preds = clf.predict(X_test).astype(y_train.to_numpy().dtype)
proba = clf.predict_proba(X_test)[:, 1]
elapsed_default = time.perf_counter() - start

accuracy_default = accuracy_score(y_test, preds)
roc_auc_default = roc_auc_score(y_test, proba)
print(f"[default]  accuracy={accuracy_default:.4f}  roc_auc={roc_auc_default:.4f}  ({elapsed_default:.1f}s)")

**Expected output** (numbers verified in this repo's own reference environment; yours should be close,
not necessarily identical, depending on CPU/GPU and library versions):
```
[default]  accuracy=0.9737  roc_auc=0.9970  (17.1s)
```

**How to verify your run succeeded:** accuracy in the 0.93–0.98 range on this dataset. If you got an
exception instead, go to [`docs/08-troubleshooting.md`](../docs/08-troubleshooting.md) — the first two
entries there cover the most common first-run failures.

## Step 4 — Try the ensemble preset

`TabFMClassifier.ensemble(model=...)` turns on several extras at once: random feature crosses, SVD-derived
features, non-negative-least-squares (NNLS) blending across ensemble members, and probability calibration
(Platt scaling). Full breakdown:
[`docs/04-core-concepts.md §5`](../docs/04-core-concepts.md#5-the-ensemble-preset-what-it-actually-turns-on).

It's usually more accurate — but not *always*, and it's always more expensive. Let's measure both, honestly,
instead of assuming.

In [ ]:
clf_ensemble = TabFMClassifier.ensemble(model=model, n_estimators=16, random_state=42)

start = time.perf_counter()
clf_ensemble.fit(X_train, y_train.to_numpy())
preds_e = clf_ensemble.predict(X_test).astype(y_train.to_numpy().dtype)
proba_e = clf_ensemble.predict_proba(X_test)[:, 1]
elapsed_ensemble = time.perf_counter() - start

accuracy_ensemble = accuracy_score(y_test, preds_e)
roc_auc_ensemble = roc_auc_score(y_test, proba_e)
print(f"[ensemble] accuracy={accuracy_ensemble:.4f}  roc_auc={roc_auc_ensemble:.4f}  ({elapsed_ensemble:.1f}s)")

**Expected output** (verified reference run, `n_estimators=16` for both variants):
```
[default]         accuracy=0.9737  roc_auc=0.9970  (17.1s)
[ensemble preset]  accuracy=0.9737  roc_auc=0.9964  (61.6s)
```

**Interpretation:** on this particular dataset, the ensemble preset took roughly **3.6x longer** and did
**not** improve accuracy or ROC-AUC — it was marginally lower on this run. This is a real, honest result,
not a mistake: `breast_cancer` is a small, well-separated dataset where the default preset already performs
near its ceiling. The ensemble preset earns its cost on harder, messier, real-world data — see
[`docs/07-evaluation.md`](../docs/07-evaluation.md) and
[`problems/problem1_telecom_churn`](../problems/problem1_telecom_churn) for cases where it wins clearly.

**The lesson:** always measure both on *your* data. Never assume the fancier preset automatically wins.

## Common mistakes to avoid

- **Skipping `stratify=y`** on a classification split — can leave a rare class entirely out of one split.
- **Not fixing `random_state`** — TabFM's ensemble mechanics involve randomness; without a seed, reruns
  won't match, which looks like a bug but isn't.
- **Feeding `clf.predict()` output straight into strict scikit-learn metrics** without the `.astype(...)`
  cast — see [`docs/08-troubleshooting.md#object-dtype-predictions`](../docs/08-troubleshooting.md#object-dtype-predictions).
- **Assuming the ensemble preset always wins** — measure it, as we just did above.
- **Scoring a huge test set at once on a small GPU** — memory scales with rows scored per call too, not
  just training context size. See
  [`docs/08-troubleshooting.md#cuda-out-of-memory`](../docs/08-troubleshooting.md#cuda-out-of-memory).

## What's next

- [`docs/05-working-with-data.md`](../docs/05-working-with-data.md) — preparing your own dataset correctly.
- [`examples/04_churn_baseline_comparison.py`](../examples/04_churn_baseline_comparison.py) — the same
  default-preset workflow against a real downloaded dataset and an XGBoost baseline.
- [`problems/`](../problems/) — eight full production-style case studies once you're ready for the
  advanced tier. See [`docs/10-next-steps.md`](../docs/10-next-steps.md) for a guided entry point.